# **Lista de Exercícios: Aula 6 - Curso de Python FEA.dev**


**Instruções:**

- Resolva cada exercício na célula de código logo abaixo do enunciado.

- Caso encontre algum erro no material ou tenha dúvidas, entre em contato com os membros de Tech para que possamos ajudá-lo.

- Se estiver com dificuldade para realizar os exercícios, reveja as aulas institucionais, pesquise suas dúvidas em fóruns técnicos ou busque outros vídeos complementares. Consultar documentações e pesquisar ativamente faz parte da rotina dentro da entidade.

- Evite o uso de Inteligência Artificial para resolver os exercícios. O objetivo desta lista é que você desenvolva raciocínio lógico e autonomia na resolução de problemas.

**Sugestões:**

- Antes de começar, é recomendado ler o **Zen of Python** (execute a célula abaixo). São os princípios que guiam a linguagem e vão te dar uma noção do que a comunidade considera um bom código.

- Tente desenvolver os nomes de variáveis e os comentários dentro de cada célula em inglês. A maior parte do conteúdo disponível da linguagem se encontra em inglês, e caso queira compartilhar algum projeto com a comunidade externa, isso facilita que outras pessoas entendam diretamente seu código.

**Sobre esta lista:**

- Nesta lista você vai trabalhar como um **cientista de dados**: carregar uma base de dados real, limpá-la, explorá-la, criar análises estatísticas e gerar visualizações.

- A base de dados utilizada é o **National Fossil Carbon Emissions 2024 v1.0** (Global Carbon Project), que contém emissões de CO₂ fóssil por país desde 1850 até 2023.

- Os exercícios são **progressivos**: cada um constrói sobre o anterior. Ao final, você terá uma análise completa e profissional da base de dados.

In [ ]:
import this

---
## Tópico 0 — Setup e Carregamento dos Dados

Antes de iniciar os exercícios, precisamos importar as bibliotecas necessárias e carregar a base de dados. Execute as células abaixo.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

---
## Tópico 1 — Carregamento e Exploração Inicial da Base de Dados

**Exercício 1 (Prático)** — A base de dados `National.Fossil.Carbon.Emissions.2024v1.0.xlsx` possui múltiplas abas (sheets). Vamos trabalhar com a aba `'Territorial Emissions'`, que contém as emissões territoriais de CO₂ fóssil por país.

Porém, essa planilha possui **cabeçalhos descritivos nas primeiras linhas** que não são dados. Observe que:
- As linhas 0 a 8 (índice 0-based) contêm metadados e notas explicativas.
- A linha 9 contém a unidade de medida (`MtC/yr`).
- A linha 10 contém os nomes dos países em MAIÚSCULO.
- A linha 11 contém os nomes dos países em formato legível.
- A partir da linha 12, começam os dados numéricos (ano e emissões).

**a)** Use `pd.read_excel()` com o parâmetro `header=None` para carregar a aba `'Territorial Emissions'` sem interpretar nenhuma linha como cabeçalho. Armazene o resultado em `df_raw`.

**b)** Exiba o `shape` do DataFrame para verificar quantas linhas e colunas existem.

**c)** Use `.iloc` para exibir as 15 primeiras linhas e as 5 primeiras colunas (`df_raw.iloc[:15, :5]`). Observe a estrutura e identifique onde estão os metadados e onde começam os dados.

In [ ]:
# a) Load the raw data without header


# b) Check shape


# c) Preview first rows and columns


**Exercício 2 (Prático)** — Agora que você entendeu a estrutura, vamos limpar a base de dados. Precisamos:

1. Extrair os nomes dos países a partir da **linha 11** (índice 11) — ela contém os nomes formatados corretamente.
2. Renomear a coluna 0 (que contém os anos) para `'Year'`.
3. Remover todas as linhas de metadados (linhas 0 a 11), mantendo apenas os dados a partir da linha 12.

**a)** Extraia os nomes dos países da linha 11 usando `df_raw.iloc[11, 1:]` e converta para uma lista. Armazene em `country_names`.

**b)** Crie um novo DataFrame `df` a partir de `df_raw.iloc[12:]` (dados a partir da linha 12). Use `.reset_index(drop=True)` para resetar o índice.

**c)** Renomeie as colunas: a coluna 0 deve se chamar `'Year'` e as colunas 1 em diante devem usar os nomes de `country_names`. **Dica:** construa a lista completa de nomes com `['Year'] + country_names`.

**d)** Exiba as 5 primeiras linhas com `.head()` e verifique se os nomes estão corretos.

In [ ]:
# a) Extract country names from row 11


# b) Create df with data rows only


# c) Rename columns


# d) Preview


**Exercício 3 (Prático)** — Vamos verificar e corrigir os **tipos de dados** (tipagem) do nosso DataFrame.

**a)** Use `df.dtypes` para verificar o tipo de cada coluna. O que você observa? As colunas numéricas estão como `object`?

**b)** Converta a coluna `'Year'` para `int` usando `.astype(int)`. Em seguida, converta todas as demais colunas (países) para `float` usando `.apply(pd.to_numeric, errors='coerce')` — isso garante que valores não numéricos virem `NaN` em vez de causar erro.

**c)** Verifique novamente os tipos com `df.dtypes` para confirmar que `'Year'` é `int64` e os países são `float64`.

**d)** Exiba o `.info()` do DataFrame para ter uma visão geral, incluindo a contagem de valores não-nulos por coluna.

In [ ]:
# a) Check dtypes


# b) Convert Year to int and countries to float


# c) Verify dtypes


# d) Display info


---
## Tópico 2 — Tratamento de Valores Ausentes

**Exercício 4 (Prático)** — Bases de dados reais quase sempre possuem valores ausentes (`NaN`). Vamos diagnosticar e tratar os valores faltantes.

**a)** Use `df.isna().sum().sum()` para contar o total de valores ausentes em todo o DataFrame. Exiba o resultado.

**b)** Selecione apenas as colunas de **países** (excluindo `'Year'` e as colunas de regiões ao final: `'Non KP Annex B'`, `'OECD'`, `'Non-OECD'`, `'EU27'`, `'Africa'`, `'Asia'`, `'Central America'`, `'Europe'`, `'Middle East'`, `'North America'`, `'Oceania'`, `'South America'`, `'Bunkers'`, `'Statistical Difference'`, `'World'`). Armazene essas colunas em uma lista chamada `region_cols` e crie `pure_country_cols` como as colunas que **não** estão em `['Year'] + region_cols`.

**c)** Calcule o **total de NaN por país** usando `df[pure_country_cols].isna().sum()`. Ordene de forma decrescente com `.sort_values(ascending=False)` e exiba os **10 países com mais valores ausentes**.

**d)** Preencha os valores ausentes das colunas de países com o valor anterior usando `.ffill()`. Se o primeiro ano for nulo, use também `.bfill()` em seguida. Confirme que não há mais `NaN` nas colunas de países.

In [ ]:
# a) Total NaN count


# b) Separate country columns from region columns


# c) Top 10 countries with most NaN


# d) Fill NaN with previous values (ffill) and verify


**Exercício 5 (Prático)** — Limpeza de Outliers

Muitas vezes, bases de dados contêm valores anômalos (outliers) causados por erros de digitação ou medição. Vamos criar uma regra simples para tratar isso: **se um valor for maior que o dobro do ano anterior, ele será considerado um outlier e substituído pelo valor do ano anterior**.

Para aplicar isso de forma eficiente no pandas, vamos usar as funções `.shift()` (para pegar o valor do ano anterior) e `np.where()` (para aplicar a regra condicionalmente).

**a)** Primeiro, importe a biblioteca numpy caso ainda não tenha feito: `import numpy as np`.

**b)** Crie um loop `for` que itere por todas as colunas de países (`pure_country_cols`).

**c)** Dentro do loop, para cada país (`col`), crie uma Series contendo os valores do ano anterior usando `df[col].shift(1)`. O primeiro valor (1850) será `NaN`, então preencha com 0 usando `.fillna(0)`. Chame essa Series de `prev_values`.

**d)** Use `np.where()` para aplicar a regra: se o valor atual (`df[col]`) for maior que `2 * prev_values` **E** o valor atual for maior que 0.1 (para evitar alarmes falsos com valores muito próximos de zero, como 0.001 para 0.003), substitua por `prev_values`. Caso contrário, mantenha o valor atual `df[col]`. Atualize a coluna `df[col]` com o resultado.

**e)** Verifique o resultado exibindo as emissões do Brasil dos primeiros 10 anos.

In [ ]:
# a) Import numpy (already done in setup)

# b) Loop through countries

    # c) Get previous year's values
    
    # d) Apply outlier rule using np.where()
    

# e) Verify Brazil's data


---
## Tópico 3 — Filtragem e Operações Booleanas

**Exercício 6 (Teórico)** — Explique com suas palavras:

**a)** Qual é a diferença entre os operadores `&`, `|` e `~` no pandas e os operadores `and`, `or` e `not` do Python puro? Por que não podemos usar `and`/`or` para filtrar DataFrames?

**b)** O que o método `.isin()` faz? Dê um exemplo de quando ele seria útil em vez de usar vários filtros com `|`.

**c)** Qual é a diferença entre `dropna()` e `fillna()`? Em que situação cada um seria mais adequado?

**Resposta:**

_Escreva sua resposta aqui._

**Exercício 7 (Prático)** — Vamos aplicar filtros e operações booleanas na base de dados para extrair informações relevantes.

Para facilitar as análises, primeiro vamos transformar o DataFrame do formato **wide** (cada país é uma coluna) para o formato **long** (cada linha é um par país-ano). Use o método `pd.melt()`.

**a)** Crie `df_countries` selecionando apenas as colunas `['Year'] + pure_country_cols`. Em seguida, use `pd.melt()` com `id_vars='Year'`, `var_name='Country'` e `value_name='Emissions_MtC'` para transformar o DataFrame. Armazene em `df_long`.

**b)** Converta as emissões de MtC (milhões de toneladas de carbono) para MtCO₂ (milhões de toneladas de CO₂) multiplicando por `3.664`. Crie uma nova coluna `'Emissions_MtCO2'`.

**c)** Filtre `df_long` para exibir apenas dados do **Brasil** no **século XXI** (ano ≥ 2000). Use os operadores `&` com parênteses.

**d)** Use `.isin()` para filtrar os **5 maiores emissores históricos**: `['China', 'USA', 'Russia', 'Germany', 'United Kingdom']`. Exiba apenas os dados do ano de **2023**.

In [ ]:
# a) Melt to long format


# b) Convert MtC to MtCO2


# c) Filter: Brazil in the 21st century


# d) Filter: Top 5 emitters in 2023 using isin()


---
## Tópico 4 — GroupBy e Agregação

**Exercício 8 (Prático)** — Vamos usar `groupby()` e `agg()` para responder perguntas analíticas sobre as emissões globais.

**a)** Agrupe `df_long` por `'Country'` e calcule a **soma total de emissões em MtCO₂** de cada país ao longo de toda a história (1850–2023). Ordene de forma decrescente e exiba os **15 maiores emissores históricos acumulados**.

**b)** Agrupe por `'Year'` e calcule a **soma global** de emissões (todos os países juntos) por ano. Armazene em `global_emissions`.

**c)** Use `.agg()` para calcular, por país, **múltiplas estatísticas simultaneamente**: `sum`, `mean`, `max` e `std` das emissões em MtCO₂. Exiba os resultados para os **5 maiores emissores**.

**d)** Agrupe por **duas colunas**: crie uma coluna `'Decade'` calculada como `(df_long['Year'] // 10) * 10` e agrupe por `['Decade', 'Country']`. Calcule a soma de emissões por década e por país. Exiba as emissões do **Brasil por década**.

In [ ]:
# a) Top 15 historical emitters (total)


# b) Global emissions by year


# c) Multiple statistics with agg()


# d) Emissions by decade for Brazil


---
## Tópico 5 — Análise Estatística e Temporal

**Exercício 9 (Prático)** — Vamos realizar uma análise estatística e temporal das emissões, utilizando métodos como `describe()`, `corr()`, `diff()` e `pct_change()`.

**a)** Selecione as emissões de `'Brazil'`, `'USA'`, `'China'`, `'India'` e `'Germany'` do DataFrame original `df` (formato wide). Armazene em `df_selected`. Use `.describe()` para exibir as estatísticas descritivas.

**b)** Calcule a **matriz de correlação** dessas 5 colunas usando `.corr()`. Quais pares de países possuem correlação mais forte? A correlação alta significa que os dois países aumentaram/diminuíram suas emissões de forma sincronizada ao longo da história.

**c)** Usando `df_selected`, aplique `.diff()` na coluna `'Brazil'` para calcular a **diferença absoluta** entre anos consecutivos. Exiba os 10 maiores aumentos anuais de emissão do Brasil.

**d)** Aplique `.pct_change()` na coluna `'Brazil'` para calcular a **variação percentual** entre anos. Em qual ano o Brasil teve o maior aumento percentual de emissões? E a maior queda?

In [ ]:
# a) Select countries and describe


# b) Correlation matrix


# c) diff() - absolute difference for Brazil


# d) pct_change() - percentage change for Brazil


**Exercício 10 (Prático)** — Vamos usar `np.where()` para criar categorias e aplicar o método `.replace()` para limpar dados textuais.

**a)** No `df_long`, use `np.where()` para criar uma coluna `'Emission_Level'` que classifique as emissões em MtCO₂:
- `'Very Low'` se ≤ 1
- `'Low'` se ≤ 10
- `'Medium'` se ≤ 100
- `'High'` se ≤ 1000
- `'Very High'` se > 1000

**Dica:** Use `np.where()` encadeado (aninhado).

**b)** Conte quantos registros existem em cada nível usando `.value_counts()`.

**c)** Use `.replace()` para substituir os nomes dos níveis de inglês para português: `'Very Low'` → `'Muito Baixo'`, `'Low'` → `'Baixo'`, etc.

**d)** Filtre apenas os registros do ano de **2023** que possuem nível `'Muito Alto'`. Quais países estão nessa categoria?

In [ ]:
# a) Create Emission_Level with np.where()


# b) Count records per level


# c) Replace English labels with Portuguese


# d) Very High emitters in 2023


---
## Tópico 6 — Visualização com Matplotlib e Seaborn

**Exercício 11 (Prático)** — Vamos criar visualizações profissionais para comunicar os resultados da nossa análise.

**a)** Use `matplotlib` para criar um **gráfico de linhas** das emissões globais totais ao longo do tempo (use `global_emissions` do Exercício 8b). Configure:
- Título: `'Emissões Globais de CO₂ Fóssil (1850–2023)'`
- Eixo x: `'Ano'`
- Eixo y: `'Emissões (MtCO₂)'`
- Tamanho da figura: `(12, 6)`

**b)** Use `matplotlib` para criar um **gráfico de linhas** comparando as emissões de `'Brazil'`, `'USA'`, `'China'`, `'India'` e `'Germany'` ao longo do tempo (use o DataFrame `df` original, formato wide). Adicione legenda.

**c)** Use `seaborn` para criar um **heatmap** da matriz de correlação calculada no Exercício 9b. Use `annot=True` para exibir os valores e `cmap='coolwarm'` para a paleta de cores.

**d)** Use `matplotlib` para criar um **gráfico de barras horizontal** dos 10 maiores emissores históricos acumulados (use o resultado do Exercício 8a). Use `plt.barh()` e ordene do maior para o menor.

In [ ]:
# a) Global emissions line chart


# b) Comparison line chart


# c) Correlation heatmap with seaborn


# d) Top 10 bar chart


**Exercício 12 (Prático)** — Vamos criar visualizações mais avançadas para completar nossa análise.

**a)** Use `matplotlib` para criar um gráfico de **área empilhada** (`ax.stackplot()`) mostrando a contribuição dos 5 maiores emissores ao longo do tempo. Configure cores distintas e legenda.

**b)** Use `seaborn` para criar um **histograma** (`sns.histplot()`) com KDE (Kernel Density Estimation) das emissões de 2023 de todos os países. Use `kde=True` e `bins=30`.

**c)** Crie um **scatter plot** (gráfico de dispersão) com `matplotlib` comparando as emissões de 2000 vs 2023 para cada país. Adicione uma linha diagonal (`y = x`) para identificar países que aumentaram (acima da linha) ou diminuíram (abaixo) suas emissões.

In [ ]:
# a) Stacked area plot


# b) Histogram with KDE


# c) Scatter plot: 2000 vs 2023


---
## Desafio Final

**Desafio** — Crie uma **análise completa da América do Sul** utilizando todos os conceitos aprendidos nesta lista.

Os países da América do Sul presentes na base são: `'Argentina'`, `'Bolivia'`, `'Brazil'`, `'Chile'`, `'Colombia'`, `'Ecuador'`, `'Guyana'`, `'Paraguay'`, `'Peru'`, `'Suriname'`, `'Trinidad and Tobago'`, `'Uruguay'`, `'Venezuela'`.

**a)** Filtre `df_long` para incluir apenas os países da América do Sul. Armazene em `df_sa`.

**b)** Use `groupby()` e `agg()` para calcular, por país, a soma total e a média de emissões em MtCO₂. Ordene pela soma.

**c)** Calcule a **participação percentual** de cada país no total da América do Sul no ano de **2023**.

**d)** Crie uma visualização com **4 subplots** (2x2) usando `fig, axes = plt.subplots(2, 2, figsize=(16, 12))`:

  1. **Gráfico de linhas**: emissões dos 5 maiores emissores da América do Sul ao longo do tempo.
  2. **Gráfico de barras**: emissões totais acumuladas por país da América do Sul.
  3. **Gráfico de pizza**: participação percentual em 2023.
  4. **Heatmap**: matriz de correlação entre os 5 maiores emissores da América do Sul.

**e)** Adicione um título geral com `fig.suptitle()` e use `plt.tight_layout()` para organizar o layout.

In [ ]:
# Start here

south_america = [
    'Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia',
    'Ecuador', 'Guyana', 'Paraguay', 'Peru', 'Suriname',
    'Trinidad and Tobago', 'Uruguay', 'Venezuela'
]

# a) Filter South America


# b) GroupBy + Agg


# c) Percentage share in 2023


# d) 4 subplots


# e) Layout
